In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.optimize import Bounds
import cma
from numpy.linalg import det
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import permutations
import os

# Create the output folder on the Desktop
desktop_path = os.path.expanduser("~/Desktop")
output_dir = os.path.join(desktop_path, "beta_lambda_VAE_linear_results")
os.makedirs(output_dir, exist_ok=True)

# Enable LaTeX rendering for labels (usetex=False to avoid requiring LaTeX installation)
plt.rc('text', usetex=False)

# Set random seed for reproducibility
np.random.seed(42)

# Dimensions and hyperparameters
n = 20  # Data dimension
m = 5   # Latent dimension
s = 5   # Number of generative variables
threshold = 1e-5  # Convergence threshold for fixed-point
max_iter = 2000   # Max iterations for fixed-point
beta_values = [1, 2, 4, 6]  # 4 beta values
lambda_values = [1, 2, 4, 6]  # 4 lambda values
sigma2 = 0.1  # Noise variance for Sigma_Z_tilde

# Generate Sigma_V (5x5 diagonal matrix for 5 independent generative variables)
sigma_v = 1 + 4 * np.random.rand(s)  # Standard deviations between 1 and 5
Sigma_V = np.diag(sigma_v**2)  # Diagonal matrix of variances

# Generate Gamma (n x s matrix)
Gamma = np.random.randn(n, s)  # Random transformation matrix

# Compute Sigma_Y using the generative model
Sigma_Y = Gamma @ Sigma_V @ Gamma.T + sigma2 * np.eye(n)
# Ensure symmetry and positive definiteness
Sigma_Y = (Sigma_Y + Sigma_Y.T) / 2
eigvals = np.linalg.eigvalsh(Sigma_Y)
if np.any(eigvals <= 0):
    Sigma_Y += (1e-6 - np.min(eigvals)) * np.eye(n)  # Adjust for positive definiteness
Sigma_Y_inv = np.linalg.inv(Sigma_Y)

# Sub-covariance matrix XV
def sub_covariance_XV(B):
    try:
        return B @ Gamma @ Sigma_V  # Shape: (m x s) = (5 x 5)
    except np.linalg.LinAlgError:
        return np.full((m, s), np.inf)

# Compute mutual information matrix for all pairs (x_i, v_j)
def compute_mutual_information_matrix(B, Sigma_W):
    try:
        # Compute covariance of X: B * Sigma_Y * B.T + Sigma_W
        Sigma_X = B @ Sigma_Y @ B.T + Sigma_W
        if det(Sigma_X) <= 0:
            return np.full((m, s), np.inf)
        
        # Sub-covariance matrix: cov(X, V) = B * Gamma * Sigma_V
        cov_XV = sub_covariance_XV(B)
        if np.any(np.isinf(cov_XV)):
            return np.full((m, s), np.inf)
        
        # Initialize mutual information matrix
        mi_matrix = np.zeros((m, s))
        for i in range(m):
            for j in range(s):
                Sigma_xi = Sigma_X[i, i]  # Variance of x_i
                Sigma_vj = Sigma_V[j, j]  # Variance of v_j
                cov_xi_vj = cov_XV[i, j]  # cov(x_i, v_j)
                Sigma_xi_vj = np.array([[Sigma_xi, cov_xi_vj],
                                       [cov_xi_vj, Sigma_vj]])
                det_xi_vj = det(Sigma_xi_vj)
                if det_xi_vj <= 0:
                    mi_matrix[i, j] = np.inf
                else:
                    mi_matrix[i, j] = 0.5 * np.log(Sigma_xi * Sigma_vj / det_xi_vj)
        return mi_matrix
    except np.linalg.LinAlgError:
        return np.full((m, s), np.inf)

# Disentanglement metric I_5 (maximum sum over all permutations)
def disentanglement_I5(B, Sigma_W):
    try:
        # Compute the mutual information matrix
        mi_matrix = compute_mutual_information_matrix(B, Sigma_W)
        if np.any(np.isinf(mi_matrix)):
            return np.inf
        
        # Compute I_5 as the maximum sum across all permutations
        max_sum = -np.inf
        for perm in permutations(range(s)):
            current_sum = sum(mi_matrix[i, pi] for i, pi in enumerate(perm))
            max_sum = max(max_sum, current_sum)
        
        return max_sum
    except np.linalg.LinAlgError:
        return np.inf

# Mutual information functions
def encoder_mi(B, Sigma_W):
    try:
        num = B @ Sigma_Y @ B.T + Sigma_W
        det_num = det(num)
        det_den = det(Sigma_W)
        if det_num <= 0 or det_den <= 0:
            return np.inf
        return 0.5 * np.log(det_num / det_den)
    except np.linalg.LinAlgError:
        return np.inf

def decoder_mi(A, Sigma_Z):
    try:
        num = A @ A.T + Sigma_Z
        det_num = det(num)
        det_den = det(Sigma_Z)
        if det_num <= 0 or det_den <= 0:
            return np.inf
        return 0.5 * np.log(det_num / det_den)
    except np.linalg.LinAlgError:
        return np.inf

# Objective function Gamma_1 with reconstruction errors
def gamma_1(A, B, Sigma_Z, Sigma_W, beta, lambda_):
    try:
        Sigma_Z_inv = np.linalg.inv(Sigma_Z)
        Sigma_W_inv = np.linalg.inv(Sigma_W)
        
        # Check determinants for positive definiteness
        det_Sigma_Z = det(Sigma_Z)
        det_Sigma_W = det(Sigma_W)
        if det_Sigma_Z <= 0 or det_Sigma_W <= 0:
            return np.inf, np.inf, np.inf
        
        # First term: Log-likelihood reconstruction error
        trace1 = np.trace(A.T @ Sigma_Z_inv @ Sigma_Y @ B.T)
        trace2 = np.trace(Sigma_Z_inv @ A @ B @ Sigma_Y)
        trace3 = np.trace(Sigma_Z_inv @ Sigma_Y)
        trace4 = np.trace(A.T @ Sigma_Z_inv @ A @ (B @ Sigma_Y @ B.T + Sigma_W))
        recon_ll = -0.5 * (trace1 + trace2 - trace3 - trace4 - \
                           n * np.log(2 * np.pi) - np.log(det_Sigma_Z))
        
        # Second term: KL divergence
        term2 = 0.5 * beta * (np.trace(B @ Sigma_Y @ B.T + Sigma_W) - \
                              np.log(det_Sigma_W) - m)
        
        # Third term: L2-norm with lambda for objective
        l2_base = np.trace(
            (np.eye(n) - A @ B) @ Sigma_Y @ (np.eye(n) - A @ B).T + \
            A @ Sigma_W @ A.T
        )
        term3 = lambda_ * l2_base
        recon_l2 = l2_base  # Unscaled L2-norm for reporting
        
        total = recon_ll + term2 + term3
        return total, recon_ll, recon_l2
    except np.linalg.LinAlgError:
        return np.inf, np.inf, np.inf

# Flatten parameters
def flatten_params(A, B, Sigma_Z, Sigma_W):
    L_W = np.linalg.cholesky(Sigma_W)
    L_W_vec = L_W[np.tril_indices(m)]
    return np.concatenate([
        A.flatten(),
        B.flatten(),
        np.diag(Sigma_Z).flatten(),
        L_W_vec.flatten()
    ])

# Unflatten parameters
def unflatten_params(x, n, m):
    idx_A = n * m
    idx_B = idx_A + m * n
    idx_Sigma_Z = idx_B + n
    A = x[:idx_A].reshape(n, m)
    B = x[idx_A:idx_B].reshape(m, n)
    diag_Z = x[idx_B:idx_Sigma_Z]
    Sigma_Z = np.diag(np.maximum(diag_Z, 1e-1))
    L_W_vec = x[idx_Sigma_Z:]
    L_W = np.zeros((m, m))
    tril_indices = np.tril_indices(m)
    L_W[tril_indices] = L_W_vec
    for i in range(m):
        L_W[i, i] = np.maximum(L_W[i, i], 2)
    Sigma_W = L_W @ L_W.T
    return A, B, Sigma_Z, Sigma_W

# Objective function for L-BFGS-B and CMA-ES
def objective(x, n, m, beta, lambda_):
    A, B, Sigma_Z, Sigma_W = unflatten_params(x, n, m)
    total, _, _ = gamma_1(A, B, Sigma_Z, Sigma_W, beta, lambda_)
    return total

# Fixed-point iteration update functions
def update_A(Sigma_Z, B, Sigma_W):
    try:
        Sigma_W_inv = np.linalg.inv(Sigma_W)
        temp = np.linalg.inv(Sigma_Y_inv + B.T @ Sigma_W_inv @ B)
        return temp @ B.T @ Sigma_W_inv
    except np.linalg.LinAlgError:
        return np.zeros((n, m))

def update_B(A, Sigma_Z, beta, lambda_):
    try:
        Sigma_Z_inv = np.linalg.inv(Sigma_Z)
        temp_matrix = Sigma_Z_inv + 2 * lambda_ * np.eye(n)
        temp_matrix = temp_matrix / beta
        temp = np.linalg.inv(np.eye(m) + A.T @ temp_matrix @ A)
        return temp @ A.T @ temp_matrix
    except np.linalg.LinAlgError:
        return np.zeros((m, n))

def update_Sigma_Z(B, Sigma_W):
    try:
        Sigma_W_inv = np.linalg.inv(Sigma_W)
        Sigma_Z = np.linalg.inv(Sigma_Y_inv + B.T @ Sigma_W_inv @ B)
        return np.diag(np.maximum(np.diag(Sigma_Z), 1e-1))
    except np.linalg.LinAlgError:
        return np.diag(np.ones(n) * 1.0)

def update_Sigma_W(A, Sigma_Z, beta, lambda_):
    try:
        Sigma_Z_inv = np.linalg.inv(Sigma_Z)
        temp_matrix = Sigma_Z_inv + 2 * lambda_ * np.eye(n)
        temp_matrix = temp_matrix / beta
        Sigma_W = np.linalg.inv(np.eye(m) + A.T @ temp_matrix @ A)
        L_W = np.linalg.cholesky(Sigma_W + 1e-1 * np.eye(m))
        return L_W @ L_W.T
    except np.linalg.LinAlgError:
        return np.eye(m) * 1.0

# Frobenius norm for convergence
def frobenius_diff(A_new, A_old, B_new, B_old, Sigma_Z_new, Sigma_Z_old, Sigma_W_new, Sigma_W_old):
    return (np.linalg.norm(A_new - A_old, 'fro') +
            np.linalg.norm(B_new - B_old, 'fro') +
            np.linalg.norm(Sigma_Z_new - Sigma_Z_old, 'fro') +
            np.linalg.norm(Sigma_W_new - Sigma_W_old, 'fro'))

# Initial parameters
A_init = 0.1 * np.random.randn(n, m)
B_init = 0.1 * np.random.randn(m, n)
Sigma_Z_init = np.diag(1 + 0.01 * np.abs(np.random.randn(n)))
Sigma_W_init = np.eye(m) + 0.01 * np.random.randn(m, m)
Sigma_W_init = (Sigma_W_init + Sigma_W_init.T) / 2 + 1e-1 * np.eye(m)

# Store results and reconstruction errors for each method
results = []
# Dictionaries to store reconstruction errors for each method as 4x4 matrices
recon_ll_matrices = {
    'analytical': np.zeros((4, 4)),
    'fixed_point': np.zeros((4, 4)),
    'lbfgs': np.zeros((4, 4)),
    'cmaes': np.zeros((4, 4))
}
recon_l2_matrices = {
    'analytical': np.zeros((4, 4)),
    'fixed_point': np.zeros((4, 4)),
    'lbfgs': np.zeros((4, 4)),
    'cmaes': np.zeros((4, 4))
}

# Loop over beta and lambda values
for lambda_idx, lambda_ in enumerate(lambda_values):
    for beta_idx, beta in enumerate(beta_values):
        print(f"\nTesting beta={beta}, lambda={lambda_}")

        # Fixed-Point Iteration
        A_fp = A_init.copy()
        B_fp = B_init.copy()
        Sigma_Z_fp = Sigma_Z_init.copy()
        Sigma_W_fp = Sigma_W_init.copy()
        for i in range(max_iter):
            A_old = A_fp.copy()
            B_old = B_fp.copy()
            Sigma_Z_old = Sigma_Z_fp.copy()
            Sigma_W_old = Sigma_W_fp.copy()
            B_fp = update_B(A_fp, Sigma_Z_fp, beta, lambda_)
            Sigma_W_fp = update_Sigma_W(A_fp, Sigma_Z_fp, beta, lambda_)
            A_fp = update_A(Sigma_Z_fp, B_fp, Sigma_W_fp)
            Sigma_Z_fp = update_Sigma_Z(B_fp, Sigma_W_fp)
            diff = frobenius_diff(A_fp, A_old, B_fp, B_old, Sigma_Z_fp, Sigma_Z_old, Sigma_W_fp, Sigma_W_old)
            if diff < threshold:
                print(f"Fixed-Point Iteration converged after {i+1} iterations")
                break
        else:
            print("Fixed-Point Iteration did not converge within max iterations")
        obj_fp, recon_ll_fp, recon_l2_fp = gamma_1(A_fp, B_fp, Sigma_Z_fp, Sigma_W_fp, beta, lambda_)
        mi_enc_fp = encoder_mi(B_fp, Sigma_W_fp)
        mi_dec_fp = decoder_mi(A_fp, Sigma_Z_fp)
        sub_cov_XV_fp = sub_covariance_XV(B_fp)
        mi_matrix_fp = compute_mutual_information_matrix(B_fp, Sigma_W_fp)
        I5_fp = disentanglement_I5(B_fp, Sigma_W_fp)

        # L-BFGS-B Optimization
        x_init = flatten_params(A_init, B_init, Sigma_Z_init, Sigma_W_init)
        lb = np.concatenate([
            -10 * np.ones(n * m),
            -10 * np.ones(m * n),
            1e-1 * np.ones(n),
            -10 * np.ones((m * (m + 1)) // 2)
        ])
        ub = np.concatenate([
            10 * np.ones(n * m),
            10 * np.ones(m * n),
            np.inf * np.ones(n),
            10 * np.ones((m * (m + 1)) // 2)
        ])
        bounds = Bounds(lb, ub)
        result_lbfgs = minimize(
            lambda x: objective(x, n, m, beta, lambda_), x_init,
            method='L-BFGS-B', bounds=bounds,
            options={'disp': False, 'maxiter': 2000, 'gtol': 1e-5}
        )
        A_lbfgs, B_lbfgs, Sigma_Z_lbfgs, Sigma_W_lbfgs = unflatten_params(result_lbfgs.x, n, m)
        obj_lbfgs, recon_ll_lbfgs, recon_l2_lbfgs = gamma_1(A_lbfgs, B_lbfgs, Sigma_Z_lbfgs, Sigma_W_lbfgs, beta, lambda_)
        mi_enc_lbfgs = encoder_mi(B_lbfgs, Sigma_W_lbfgs)
        mi_dec_lbfgs = decoder_mi(A_lbfgs, Sigma_Z_lbfgs)
        sub_cov_XV_lbfgs = sub_covariance_XV(B_lbfgs)
        mi_matrix_lbfgs = compute_mutual_information_matrix(B_lbfgs, Sigma_W_lbfgs)
        I5_lbfgs = disentanglement_I5(B_lbfgs, Sigma_W_lbfgs)

        # CMA-ES Optimization
        opts = {
            'bounds': [lb, ub],
            'tolfun': 1e-8,
            'maxiter': 2000,
            'maxfevals': 50000,
            'popsize': 100,
            'verb_disp': 0
        }
        es = cma.CMAEvolutionStrategy(x_init, 0.1, opts)
        es.optimize(lambda x: objective(x, n, m, beta, lambda_))
        x_cmaes = es.result.xbest
        A_cmaes, B_cmaes, Sigma_Z_cmaes, Sigma_W_cmaes = unflatten_params(x_cmaes, n, m)
        obj_cmaes, recon_ll_cmaes, recon_l2_cmaes = gamma_1(A_cmaes, B_cmaes, Sigma_Z_cmaes, Sigma_W_cmaes, beta, lambda_)
        mi_enc_cmaes = encoder_mi(B_cmaes, Sigma_W_cmaes)
        mi_dec_cmaes = decoder_mi(A_cmaes, Sigma_Z_cmaes)
        sub_cov_XV_cmaes = sub_covariance_XV(B_cmaes)
        mi_matrix_cmaes = compute_mutual_information_matrix(B_cmaes, Sigma_W_cmaes)
        I5_cmaes = disentanglement_I5(B_cmaes, Sigma_W_cmaes)

        # Analytical solution
        A_opt = np.linalg.inv(Sigma_Y_inv + B_fp.T @ np.linalg.inv(Sigma_W_fp) @ B_fp) @ B_fp.T @ np.linalg.inv(Sigma_W_fp)
        B_opt = np.linalg.inv(np.eye(m) + A_fp.T @ (np.linalg.inv(Sigma_Z_fp) + 2 * lambda_ * np.eye(n)) / beta @ A_fp) @ A_fp.T @ (np.linalg.inv(Sigma_Z_fp) + 2 * lambda_ * np.eye(n)) / beta
        Sigma_Z_opt = np.diag(np.maximum(np.diag(np.linalg.inv(Sigma_Y_inv + B_fp.T @ np.linalg.inv(Sigma_W_fp) @ B_fp)), 1e-1))
        Sigma_W_opt = np.linalg.inv(np.eye(m) + A_fp.T @ (np.linalg.inv(Sigma_Z_fp) + 2 * lambda_ * np.eye(n)) / beta @ A_fp)
        L_W_opt = np.linalg.cholesky(Sigma_W_opt + 1e-1 * np.eye(m))
        Sigma_W_opt = L_W_opt @ L_W_opt.T
        obj_opt, recon_ll_opt, recon_l2_opt = gamma_1(A_opt, B_opt, Sigma_Z_opt, Sigma_W_opt, beta, lambda_)
        mi_enc_opt = encoder_mi(B_opt, Sigma_W_opt)
        mi_dec_opt = decoder_mi(A_opt, Sigma_Z_opt)
        sub_cov_XV_opt = sub_covariance_XV(B_opt)
        mi_matrix_opt = compute_mutual_information_matrix(B_opt, Sigma_W_opt)
        I5_opt = disentanglement_I5(B_opt, Sigma_W_opt)

        # Store reconstruction errors in 6x6 matrices
        recon_ll_matrices['analytical'][lambda_idx, beta_idx] = recon_ll_opt
        recon_ll_matrices['fixed_point'][lambda_idx, beta_idx] = recon_ll_fp
        recon_ll_matrices['lbfgs'][lambda_idx, beta_idx] = recon_ll_lbfgs
        recon_ll_matrices['cmaes'][lambda_idx, beta_idx] = recon_ll_cmaes

        recon_l2_matrices['analytical'][lambda_idx, beta_idx] = recon_l2_opt
        recon_l2_matrices['fixed_point'][lambda_idx, beta_idx] = recon_l2_fp
        recon_l2_matrices['lbfgs'][lambda_idx, beta_idx] = recon_l2_lbfgs
        recon_l2_matrices['cmaes'][lambda_idx, beta_idx] = recon_l2_cmaes

        # Generate and save mutual information heatmaps for each method with blueish colormap
        for method, mi_matrix in [
            ('analytical', mi_matrix_opt),
            ('fixed_point', mi_matrix_fp),
            ('lbfgs', mi_matrix_lbfgs),
            ('cmaes', mi_matrix_cmaes)
        ]:
            plt.figure(figsize=(8, 6))
            if np.all(np.isfinite(mi_matrix)):
                sns.heatmap(
                    mi_matrix,
                    annot=True, fmt=".2f", cmap='Blues',  # Blueish colormap
                    xticklabels=[r'$v_{' + str(i+1) + '}$' for i in range(s)],
                    yticklabels=[r'$x_{' + str(i+1) + '}$' for i in range(m)],
                    cbar_kws={'label': 'Mutual Information (nats)'},
                    annot_kws={"size": 16}  # Increased font size for annotations
                )
                plt.title(f"Mutual Information Heatmap (β={beta}, λ={lambda_}, Method={method})")
                plt.xlabel("Generative Variables")
                plt.ylabel("Latent Variables")
            else:
                plt.text(0.5, 0.5, "Invalid Data (contains inf or nan)",
                         horizontalalignment='center', verticalalignment='center')
                plt.title(f"Mutual Information Heatmap (β={beta}, λ={lambda_}, Method={method})")
                plt.xlabel("Generative Variables")
                plt.ylabel("Latent Variables")
            plt.savefig(os.path.join(output_dir, f"heatmap_beta{beta}_lambda{lambda_}_{method}.png"))
            plt.close()

        # Store results
        result = {
            'beta': beta,
            'lambda': lambda_,
            'analytical': {
                'obj': obj_opt,
                'recon_ll': recon_ll_opt,
                'recon_l2': recon_l2_opt,
                'mi_enc': mi_enc_opt,
                'mi_dec': mi_dec_opt,
                'sub_cov_XV': sub_cov_XV_opt,
                'I5': I5_opt,
                'mi_matrix': mi_matrix_opt
            },
            'fixed_point': {
                'obj': obj_fp,
                'recon_ll': recon_ll_fp,
                'recon_l2': recon_l2_fp,
                'mi_enc': mi_enc_fp,
                'mi_dec': mi_dec_fp,
                'sub_cov_XV': sub_cov_XV_fp,
                'I5': I5_fp,
                'mi_matrix': mi_matrix_fp
            },
            'lbfgs': {
                'obj': obj_lbfgs,
                'recon_ll': recon_ll_lbfgs,
                'recon_l2': recon_l2_lbfgs,
                'mi_enc': mi_enc_lbfgs,
                'mi_dec': mi_dec_lbfgs,
                'sub_cov_XV': sub_cov_XV_lbfgs,
                'I5': I5_lbfgs,
                'mi_matrix': mi_matrix_lbfgs
            },
            'cmaes': {
                'obj': obj_cmaes,
                'recon_ll': recon_ll_cmaes,
                'recon_l2': recon_l2_cmaes,
                'mi_enc': mi_enc_cmaes,
                'mi_dec': mi_dec_cmaes,
                'sub_cov_XV': sub_cov_XV_cmaes,
                'I5': I5_cmaes,
                'mi_matrix': mi_matrix_cmaes
            },
            'param_diff_lbfgs': (
                np.linalg.norm(A_lbfgs - A_opt, 'fro') +
                np.linalg.norm(B_lbfgs - B_opt, 'fro') +
                np.linalg.norm(Sigma_Z_lbfgs - Sigma_Z_opt, 'fro') +
                np.linalg.norm(Sigma_W_lbfgs - Sigma_W_opt, 'fro')
            ),
            'param_diff_cmaes': (
                np.linalg.norm(A_cmaes - A_opt, 'fro') +
                np.linalg.norm(B_cmaes - B_opt, 'fro') +
                np.linalg.norm(Sigma_Z_cmaes - Sigma_Z_opt, 'fro') +
                np.linalg.norm(Sigma_W_cmaes - Sigma_W_opt, 'fro')
            )
        }
        results.append(result)

# Generate and save reconstruction error heatmaps for each method with redish colormap
for method in ['analytical', 'fixed_point', 'lbfgs', 'cmaes']:
    # Log-Likelihood Reconstruction Error Heatmap
    recon_ll_matrix = recon_ll_matrices[method]
    plt.figure(figsize=(8, 8))
    if np.all(np.isfinite(recon_ll_matrix)):
        sns.heatmap(
            recon_ll_matrix,
            annot=True, fmt=".2f", cmap='Reds',  # Redish colormap
            xticklabels=[f"{beta}" for beta in beta_values],
            yticklabels=[f"{lambda_}" for lambda_ in lambda_values],
            cbar_kws={'label': 'Log-Likelihood Reconstruction Error'},
            annot_kws={"size": 16}  # Increased font size for annotations
        )
        plt.title(f"Log-Likelihood Error Heatmap (Method={method})")
        plt.xlabel("β")
        plt.ylabel("λ")
    else:
        plt.text(0.5, 0.5, "Invalid Data (contains inf or nan)",
                 horizontalalignment='center', verticalalignment='center')
        plt.title(f"Log-Likelihood Error Heatmap (Method={method})")
        plt.xlabel("β")
        plt.ylabel("λ")
    plt.savefig(os.path.join(output_dir, f"heatmap_recon_ll_{method}.png"))
    plt.close()

    # L2-Norm Reconstruction Error Heatmap
    recon_l2_matrix = recon_l2_matrices[method]
    plt.figure(figsize=(8, 8))
    if np.all(np.isfinite(recon_l2_matrix)):
        sns.heatmap(
            recon_l2_matrix,
            annot=True, fmt=".2f", cmap='Reds',  # Redish colormap
            xticklabels=[f"{beta}" for beta in beta_values],
            yticklabels=[f"{lambda_}" for lambda_ in lambda_values],
            cbar_kws={'label': 'L2-Norm Reconstruction Error'},
            annot_kws={"size": 16}  # Increased font size for annotations
        )
        plt.title(f"L2-Norm Error Heatmap (Method={method})")
        plt.xlabel("β")
        plt.ylabel("λ")
    else:
        plt.text(0.5, 0.5, "Invalid Data (contains inf or nan)",
                 horizontalalignment='center', verticalalignment='center')
        plt.title(f"L2-Norm Error Heatmap (Method={method})")
        plt.xlabel("β")
        plt.ylabel("λ")
    plt.savefig(os.path.join(output_dir, f"heatmap_recon_l2_{method}.png"))
    plt.close()

# Print results with 2 decimal places
print("\nFinal Results:")
for result in results:
    beta = result['beta']
    lambda_ = result['lambda']
    print(f"\nBeta={beta}, Lambda={lambda_}")
    print(f"{'Method':<15} {'Objective':>12} {'Log-Likelihood':>15} {'L2-Norm':>12} {'Encoder MI':>12} {'Decoder MI':>12} {'I5':>12}")
    print("-" * 95)
    for method in ['analytical', 'fixed_point', 'lbfgs', 'cmaes']:
        data = result[method]
        obj = data['obj']
        recon_ll = data['recon_ll']
        recon_l2 = data['recon_l2']
        mi_enc = data['mi_enc']
        mi_dec = data['mi_dec']
        I5 = data['I5']
        print(f"{method.replace('_', ' ').title():<15} {obj:>12.2f} {recon_ll:>15.2f} {recon_l2:>12.2f} {mi_enc:>12.2f} {mi_dec:>12.2f} {I5:>12.2f}")
    print(f"\nObjective Differences (vs Analytical):")
    print(f"Fixed-Point: {abs(result['fixed_point']['obj'] - result['analytical']['obj']):.2f}")
    print(f"L-BFGS-B: {abs(result['lbfgs']['obj'] - result['analytical']['obj']):.2f}")
    print(f"CMA-ES: {abs(result['cmaes']['obj'] - result['analytical']['obj']):.2f}")
    print(f"\nFrobenius Norm of Parameter Differences (vs Analytical):")
    print(f"L-BFGS-B: {result['param_diff_lbfgs']:.2f}")
    print(f"CMA-ES: {result['param_diff_cmaes']:.2f}")


Testing beta=1, lambda=1
Fixed-Point Iteration converged after 359 iterations

Testing beta=2, lambda=1
Fixed-Point Iteration converged after 237 iterations

Testing beta=4, lambda=1
Fixed-Point Iteration converged after 150 iterations

Testing beta=6, lambda=1
Fixed-Point Iteration converged after 111 iterations

Testing beta=1, lambda=2
Fixed-Point Iteration converged after 447 iterations

Testing beta=2, lambda=2
Fixed-Point Iteration converged after 309 iterations

Testing beta=4, lambda=2
Fixed-Point Iteration converged after 210 iterations

Testing beta=6, lambda=2
Fixed-Point Iteration converged after 163 iterations

Testing beta=1, lambda=4
Fixed-Point Iteration converged after 562 iterations

Testing beta=2, lambda=4
Fixed-Point Iteration converged after 404 iterations

Testing beta=4, lambda=4
Fixed-Point Iteration converged after 286 iterations

Testing beta=6, lambda=4
Fixed-Point Iteration converged after 231 iterations

Testing beta=1, lambda=6
Fixed-Point Iteration conv